# 模型注册教程

> **前置知识**: Python基础、实验追踪概念（参见 01_ExperimentTracking_tutorial）
>
> **学习目标**: 掌握模型版本管理和生命周期控制

---

## 为什么需要模型注册？

```
生产环境的痛点:
┌─────────────────────────────────────────────────────────────┐
│  "线上跑的是哪个版本的模型？"     → 不知道                  │
│  "新模型有问题，能回滚吗？"       → 找不到旧版本            │
│  "这个模型是谁训练的？用什么数据？" → 没有记录              │
└─────────────────────────────────────────────────────────────┘

模型注册解决方案:
┌─────────────────────────────────────────────────────────────┐
│  版本管理 → 每个模型有唯一版本号，可追溯                    │
│  生命周期 → Development → Staging → Production → Archived  │
│  元数据   → 记录指标、参数、训练信息                        │
│  回滚能力 → 一键切换到任意历史版本                          │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **模型阶段** - 理解生命周期状态
2. **ModelVersion** - 版本数据结构
3. **ModelRegistry** - 注册中心核心操作
4. **阶段转换** - 推送到生产/回滚
5. **搜索比较** - 查找和对比模型
6. **MLflow 集成** - 企业级方案

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 添加源码路径，使得可以导入 src 目录下的模块
import sys
import os
sys.path.insert(0, '../src')

# 标准库
import time
import tempfile  # 创建临时目录，用于演示
import shutil    # 文件操作，用于清理
from pathlib import Path

# 导入模型注册模块
from model_registry import (
    ModelStage,        # 模型阶段枚举：DEVELOPMENT, STAGING, PRODUCTION, ARCHIVED
    ModelVersion,      # 模型版本数据类：存储版本的所有信息
    RegisteredModel,   # 注册模型数据类：存储模型的所有版本
    ModelRegistry,     # 本地注册中心：基于文件系统
    create_registry,   # 工厂函数：创建注册中心
    MLFLOW_AVAILABLE,  # MLflow 是否可用
)

# 检查可选依赖
print("=" * 50)
print("环境检查")
print("=" * 50)
print(f"MLflow 可用: {MLFLOW_AVAILABLE}")
print(f"Python 版本: {sys.version.split()[0]}")

## 1. 模型阶段 (ModelStage)

**核心概念**: 模型在生命周期中会经历不同的阶段，每个阶段有不同的用途

```
模型生命周期:
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│ Development │ →  │   Staging   │ →  │ Production  │ →  │  Archived   │
│   (开发)    │    │  (预发布)   │    │   (生产)    │    │  (归档)     │
└─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘
      │                  │                  │                  │
      ▼                  ▼                  ▼                  ▼
   单元测试          集成测试           A/B测试            保留备份
   代码审查          性能测试           全量上线           可回滚
```

**阶段说明**:
- **NONE**: 未分配阶段（刚注册的模型）
- **DEVELOPMENT**: 开发中，可能不稳定
- **STAGING**: 预发布，在测试环境验证
- **PRODUCTION**: 生产环境，服务真实用户
- **ARCHIVED**: 已归档，保留用于回滚

In [ ]:
# ============================================================
# 查看模型阶段枚举
# ============================================================
# ModelStage 是一个枚举类，定义了模型的生命周期状态
# 使用枚举而不是字符串可以防止拼写错误，并提供IDE自动补全

print("=" * 50)
print("模型阶段类型")
print("=" * 50)
for stage in ModelStage:
    print(f"  {stage.name:15} → {stage.value}")

print("\n" + "-" * 50)
print("阶段转换规则:")
print("  1. 新注册的模型默认是 NONE 阶段")
print("  2. 推送到 PRODUCTION 时，旧的 PRODUCTION 版本自动归档")
print("  3. ARCHIVED 版本可以重新激活（用于回滚）")

## 2. 模型版本数据结构

**核心概念**: `ModelVersion` 存储单个模型版本的所有信息

```
ModelVersion 数据结构:
┌─────────────────────────────────────────────────────────────┐
│  name: str              ← 模型名称（用于分组）              │
│  version: str           ← 版本号（如 "1.0.0"）              │
│  stage: ModelStage      ← 当前阶段                          │
│  model_path: str        ← 模型文件路径                      │
│  metrics: Dict          ← 性能指标（accuracy, loss 等）     │
│  params: Dict           ← 训练参数（lr, epochs 等）         │
│  tags: Dict             ← 标签（用于分类和搜索）            │
│  description: str       ← 版本描述                          │
│  created_at: float      ← 创建时间戳                        │
└─────────────────────────────────────────────────────────────┘
```

**版本号规范** (语义化版本):
```
主版本.次版本.修订号
  │       │      │
  │       │      └── 修复bug，不影响功能
  │       └────────── 新增功能，向后兼容
  └────────────────── 重大变更，不向后兼容
```

# ============================================================
# 创建 ModelVersion 实例
# ============================================================
# ModelVersion 是一个 dataclass，存储单个模型版本的所有信息
# 包括模型文件路径、性能指标、训练参数、标签等

version = ModelVersion(
    name="image_classifier",           # 模型名称：用于分组相关版本
    version="1.0.0",                   # 版本号：推荐使用语义化版本
    stage=ModelStage.STAGING,          # 当前阶段：预发布
    model_path="/models/classifier_v1.pt",  # 模型文件路径
    metrics={                          # 性能指标：用于比较和选择
        "accuracy": 0.95,
        "f1_score": 0.93,
        "inference_time_ms": 15.2
    },
    params={                           # 训练参数：用于复现
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 32
    },
    tags={                             # 标签：用于分类和搜索
        "framework": "pytorch",
        "task": "classification",
        "dataset": "imagenet"
    },
    description="ResNet50 图像分类模型 - 基线版本"
)

# 查看模型版本信息
print("=" * 50)
print("模型版本信息")
print("=" * 50)
print(f"名称:   {version.name}")
print(f"版本:   {version.version}")
print(f"阶段:   {version.stage.value}")
print(f"路径:   {version.model_path}")
print(f"描述:   {version.description}")
print("\n指标:")
for k, v in version.metrics.items():
    print(f"  {k}: {v}")
print("\n参数:")
for k, v in version.params.items():
    print(f"  {k}: {v}")

In [ ]:
## 3. ModelRegistry 模型注册中心

**核心概念**: `ModelRegistry` 是管理所有模型版本的中心系统

```
ModelRegistry 核心操作:
┌─────────────────────────────────────────────────────────────┐
│  register_model()     → 注册新版本                          │
│  get_model()          → 获取模型（按版本/阶段）             │
│  transition_stage()   → 转换阶段（推送/回滚）               │
│  list_models()        → 列出所有模型                        │
│  list_versions()      → 列出模型的所有版本                  │
│  search_models()      → 搜索模型（按名称/标签/阶段）        │
│  compare_versions()   → 比较两个版本的差异                  │
│  delete_model()       → 删除模型或版本                      │
└─────────────────────────────────────────────────────────────┘

文件结构:
model_registry/
├── image_classifier/
│   ├── 1.0.0/
│   │   ├── model.pkl
│   │   └── metadata.json
│   └── 2.0.0/
│       ├── model.pkl
│       └── metadata.json
└── text_encoder/
    └── 1.0.0/
        ├── model.pkl
        └── metadata.json
```

# ============================================================
# 创建 ModelRegistry 实例
# ============================================================
# ModelRegistry 是模型注册中心的核心类
# 它管理所有模型的版本、阶段和元数据

# 创建临时目录用于演示（实际使用时指定固定目录）
demo_dir = tempfile.mkdtemp(prefix="registry_demo_")
print(f"演示目录: {demo_dir}")

# 创建注册中心
# registry_path: 注册中心的存储路径
registry = ModelRegistry(registry_path=demo_dir)

print("\n" + "=" * 50)
print("注册中心创建成功!")
print("=" * 50)
print(f"存储路径: {registry.registry_path}")

In [ ]:
### 3.1 注册模型 (register_model)

**核心操作**: 将训练好的模型注册到注册中心

```python
# register_model() 参数说明
registry.register_model(
    name="模型名称",           # 必需：用于分组相关版本
    model_path="模型文件路径",  # 必需：模型文件的路径
    version="版本号",          # 可选：不指定则自动递增
    metrics={...},            # 可选：性能指标
    params={...},             # 可选：训练参数
    tags={...},               # 可选：标签
    description="描述"        # 可选：版本描述
)
```

# ============================================================
# 注册第一个模型版本
# ============================================================
# 首先创建一个测试模型文件（实际使用时是真实的模型文件）
model_file = Path(demo_dir) / "test_model.pkl"
model_file.write_bytes(b"fake model content for demo")  # 模拟模型内容

# 注册模型版本 1.0
# register_model() 会：
# 1. 复制模型文件到注册中心
# 2. 创建元数据记录
# 3. 返回 ModelVersion 对象
v1 = registry.register_model(
    name="image_classifier",           # 模型名称
    model_path=str(model_file),        # 模型文件路径
    version="1.0",                     # 版本号
    metrics={                          # 性能指标
        "accuracy": 0.92,
        "loss": 0.25,
        "f1_score": 0.90
    },
    params={                           # 训练参数
        "lr": 0.001,
        "epochs": 50,
        "batch_size": 32
    },
    tags={"framework": "pytorch"},     # 标签
    description="初始版本 - 基线模型"   # 描述
)

print("=" * 50)
print("注册模型版本 1.0")
print("=" * 50)
print(f"名称:   {v1.name}")
print(f"版本:   {v1.version}")
print(f"阶段:   {v1.stage.value}")
print(f"指标:   {v1.metrics}")

In [ ]:
# ============================================================
# 注册更多版本（模拟迭代改进）
# ============================================================
# 在实际项目中，每次训练改进后都会注册新版本
# 这样可以追踪模型的演进历史

# 版本 2.0：更长训练时间
v2 = registry.register_model(
    name="image_classifier",
    model_path=str(model_file),
    version="2.0",
    metrics={"accuracy": 0.95, "loss": 0.18, "f1_score": 0.94},
    params={"lr": 0.0005, "epochs": 100, "batch_size": 32},
    description="改进版本 - 更长训练时间，降低学习率"
)

# 版本 3.0：数据增强
v3 = registry.register_model(
    name="image_classifier",
    model_path=str(model_file),
    version="3.0",
    metrics={"accuracy": 0.96, "loss": 0.15, "f1_score": 0.95},
    params={"lr": 0.0001, "epochs": 150, "batch_size": 64},
    description="最新版本 - 添加数据增强，更大批次"
)

print("=" * 60)
print("已注册 3 个版本")
print("=" * 60)
print(f"{'版本':<10} {'准确率':<12} {'损失':<12} {'描述'}")
print("-" * 60)
for v in [v1, v2, v3]:
    print(f"{v.version:<10} {v.metrics['accuracy']:<12} {v.metrics['loss']:<12} {v.description}")

In [ ]:
# ============================================================
# 注册另一个模型（不同类型）
# ============================================================
# 注册中心可以管理多个不同的模型
# 每个模型有自己的版本序列

text_model = registry.register_model(
    name="text_encoder",               # 不同的模型名称
    model_path=str(model_file),
    version="1.0",
    metrics={"perplexity": 15.2, "bleu": 0.85},
    params={"hidden_size": 768, "num_layers": 12},
    tags={"framework": "transformers", "task": "encoding"},
    description="BERT 文本编码器"
)

print("=" * 50)
print("注册了另一个模型: text_encoder")
print("=" * 50)
print(f"名称:   {text_model.name}")
print(f"版本:   {text_model.version}")
print(f"指标:   {text_model.metrics}")

In [ ]:
### 3.2 获取模型 (get_model)

**两种获取方式**:
1. **按版本号**: `get_model("name", version="1.0")` - 获取特定版本
2. **按阶段**: `get_model("name", stage=ModelStage.PRODUCTION)` - 获取生产版本
3. **最新版本**: `get_model("name")` - 不指定则返回最新版本

# ============================================================
# 获取模型的不同方式
# ============================================================

# 方式1: 按版本号获取
model_v2 = registry.get_model("image_classifier", version="2.0")
print("=" * 50)
print("方式1: 按版本号获取")
print("=" * 50)
print(f"获取版本 2.0:")
print(f"  准确率: {model_v2.metrics['accuracy']}")
print(f"  损失:   {model_v2.metrics['loss']}")

# 方式2: 获取最新版本（不指定版本号）
latest = registry.get_model("image_classifier")
print("\n" + "=" * 50)
print("方式2: 获取最新版本")
print("=" * 50)
print(f"最新版本: {latest.version}")
print(f"  准确率: {latest.metrics['accuracy']}")

# 注意：此时还没有设置阶段，所以按阶段获取会返回 None
print("\n提示: 新注册的模型默认是 NONE 阶段，需要手动转换到其他阶段")

In [ ]:
### 3.3 阶段转换 (transition_stage)

**核心操作**: 将模型从一个阶段转换到另一个阶段

```
典型的阶段转换流程:
┌─────────────────────────────────────────────────────────────┐
│  1. 开发完成 → transition_stage(DEVELOPMENT)               │
│  2. 测试通过 → transition_stage(STAGING)                   │
│  3. A/B测试通过 → transition_stage(PRODUCTION)             │
│  4. 新版本上线 → 旧版本自动 transition_stage(ARCHIVED)     │
└─────────────────────────────────────────────────────────────┘
```

**重要规则**: 同一模型只能有一个 PRODUCTION 版本，推送新版本时旧版本自动归档

# ============================================================
# 阶段转换示例
# ============================================================
# transition_stage() 将模型从一个阶段转换到另一个阶段
# 参数: 模型名称, 版本号, 目标阶段

# 将版本 2.0 推送到生产环境
registry.transition_stage("image_classifier", "2.0", ModelStage.PRODUCTION)
print("=" * 50)
print("阶段转换: 版本 2.0 → PRODUCTION")
print("=" * 50)

# 将版本 3.0 推送到预发布环境
registry.transition_stage("image_classifier", "3.0", ModelStage.STAGING)
print("阶段转换: 版本 3.0 → STAGING")

# 查看各版本状态
print("\n" + "-" * 50)
print("各版本当前状态:")
print("-" * 50)
for v in registry.list_versions("image_classifier"):
    print(f"  v{v.version}: {v.stage.value}")

In [ ]:
# ============================================================
# 按阶段获取模型
# ============================================================
# 在生产环境中，通常按阶段获取模型而不是按版本号
# 这样可以在不修改代码的情况下切换模型版本

prod_model = registry.get_model("image_classifier", stage=ModelStage.PRODUCTION)
staging_model = registry.get_model("image_classifier", stage=ModelStage.STAGING)

print("=" * 50)
print("按阶段获取模型")
print("=" * 50)
print(f"生产版本:   v{prod_model.version} (accuracy={prod_model.metrics['accuracy']})")
print(f"预发布版本: v{staging_model.version} (accuracy={staging_model.metrics['accuracy']})")
print("\n提示: 生产代码中推荐使用 stage 参数获取模型")
print("      这样更新模型只需要转换阶段，无需修改代码")

In [ ]:
# ============================================================
# 自动归档演示
# ============================================================
# 当推送新版本到 PRODUCTION 时，旧的 PRODUCTION 版本会自动归档
# 这确保了同一时间只有一个生产版本

print("=" * 50)
print("自动归档演示")
print("=" * 50)
print("当前生产版本: v2.0")
print("\n将 v3.0 推送到生产...")

# 推送 v3.0 到生产
registry.transition_stage("image_classifier", "3.0", ModelStage.PRODUCTION)

print("\n推送后各版本状态:")
print("-" * 50)
for v in registry.list_versions("image_classifier"):
    status_mark = "★" if v.stage == ModelStage.PRODUCTION else " "
    print(f"  {status_mark} v{v.version}: {v.stage.value}")

print("\n注意: v2.0 已自动从 PRODUCTION 归档到 ARCHIVED")

In [ ]:
### 3.4 列出模型和版本

**查询操作**:
- `list_models()`: 列出注册中心中的所有模型
- `list_versions(name)`: 列出指定模型的所有版本

# ============================================================
# 列出所有模型
# ============================================================
# list_models() 返回注册中心中所有模型的列表

models = registry.list_models()

print("=" * 50)
print("注册中心中的所有模型")
print("=" * 50)
for model in models:
    versions = registry.list_versions(model.name)
    prod_version = next((v for v in versions if v.stage == ModelStage.PRODUCTION), None)
    prod_info = f"(生产版本: v{prod_version.version})" if prod_version else "(无生产版本)"
    print(f"  {model.name}: {len(versions)} 个版本 {prod_info}")

In [ ]:
# ============================================================
# 列出模型的所有版本
# ============================================================
# list_versions() 返回指定模型的所有版本，按版本号排序

versions = registry.list_versions("image_classifier")

print("=" * 70)
print("image_classifier 版本详情")
print("=" * 70)
print(f"{'版本':<10} {'阶段':<15} {'准确率':<10} {'损失':<10} {'描述'}")
print("-" * 70)
for v in versions:
    acc = v.metrics.get('accuracy', 'N/A')
    loss = v.metrics.get('loss', 'N/A')
    # 标记生产版本
    stage_str = f"★{v.stage.value}" if v.stage == ModelStage.PRODUCTION else v.stage.value
    print(f"{v.version:<10} {stage_str:<15} {acc:<10} {loss:<10} {v.description}")

In [ ]:
### 3.5 搜索模型 (search_models)

**搜索方式**:
- 按名称搜索: `search_models(query="image")`
- 按标签搜索: `search_models(tags={"framework": "pytorch"})`
- 按阶段搜索: `search_models(stage=ModelStage.PRODUCTION)`

# ============================================================
# 搜索模型示例
# ============================================================
# search_models() 支持多种搜索方式，可以组合使用

print("=" * 50)
print("搜索模型示例")
print("=" * 50)

# 方式1: 按名称搜索（模糊匹配）
results = registry.search_models(query="image")
print(f"\n按名称搜索 'image': 找到 {len(results)} 个结果")
for r in results:
    print(f"  - {r.name} v{r.version}")

# 方式2: 按标签搜索
results = registry.search_models(tags={"framework": "pytorch"})
print(f"\n按标签搜索 framework=pytorch: 找到 {len(results)} 个结果")
for r in results:
    print(f"  - {r.name} v{r.version}")

# 方式3: 按阶段搜索
results = registry.search_models(stage=ModelStage.PRODUCTION)
print(f"\n按阶段搜索 PRODUCTION: 找到 {len(results)} 个结果")
for r in results:
    print(f"  - {r.name} v{r.version}")

In [ ]:
### 3.6 比较版本 (compare_versions)

**实用功能**: 对比两个版本的指标和参数差异，帮助决策是否升级

# ============================================================
# 比较两个版本
# ============================================================
# compare_versions() 返回两个版本的指标和参数差异
# 这对于决定是否升级模型非常有用

comparison = registry.compare_versions("image_classifier", "1.0", "3.0")

print("=" * 60)
print("版本比较: v1.0 vs v3.0")
print("=" * 60)

# 指标差异
print("\n【指标差异】")
print("-" * 60)
print(f"{'指标':<15} {'v1.0':<12} {'v3.0':<12} {'变化':<15}")
print("-" * 60)
for metric, diff in comparison["metrics_diff"].items():
    v1_val = diff.get('v1', 'N/A')
    v2_val = diff.get('v2', 'N/A')
    pct = diff.get('pct_change')
    pct_str = f"({pct:+.1f}%)" if pct else ""
    print(f"{metric:<15} {v1_val:<12} {v2_val:<12} {pct_str:<15}")

# 参数差异
print("\n【参数差异】")
print("-" * 60)
print(f"{'参数':<15} {'v1.0':<12} {'v3.0':<12}")
print("-" * 60)
for param, diff in comparison["params_diff"].items():
    print(f"{param:<15} {str(diff['v1']):<12} {str(diff['v2']):<12}")

print("\n结论: v3.0 在所有指标上都有提升，建议升级")

In [ ]:
### 3.7 更新模型信息

**维护操作**: 更新模型的描述和标签，便于管理和搜索

# ============================================================
# 更新模型信息
# ============================================================
# 可以更新模型的描述和标签，便于管理和搜索

# 更新描述
registry.update_model_description(
    "image_classifier",
    "生产级图像分类模型 - ResNet50架构，支持1000类ImageNet分类",
    version="3.0"
)

# 添加/更新标签
registry.set_model_tags(
    "image_classifier",
    {
        "team": "cv",           # 负责团队
        "priority": "high",     # 优先级
        "use_case": "product",  # 使用场景
        "reviewed": "true"      # 是否已审核
    },
    version="3.0"
)

# 查看更新后的信息
model = registry.get_model("image_classifier", version="3.0")
print("=" * 50)
print("更新后的模型信息")
print("=" * 50)
print(f"描述: {model.description}")
print(f"\n标签:")
for k, v in model.tags.items():
    print(f"  {k}: {v}")

In [ ]:
### 3.8 删除模型 (delete_model)

**清理操作**: 删除不再需要的模型或版本

⚠️ **注意**: 删除操作不可逆，请谨慎使用

# ============================================================
# 删除模型版本
# ============================================================
# delete_model() 可以删除特定版本或整个模型
# 注意：删除操作不可逆，请谨慎使用

print("=" * 50)
print("删除模型版本")
print("=" * 50)

# 删除前查看版本列表
print("删除前的版本:")
for v in registry.list_versions("image_classifier"):
    print(f"  v{v.version}: {v.stage.value}")

# 删除特定版本（通常删除旧的归档版本）
registry.delete_model("image_classifier", version="1.0")
print("\n已删除版本 1.0")

# 删除后查看版本列表
print("\n删除后的版本:")
for v in registry.list_versions("image_classifier"):
    print(f"  v{v.version}: {v.stage.value}")

print("\n提示: 建议只删除已归档的旧版本，保留生产版本用于回滚")

In [ ]:
## 4. 工厂函数 (create_registry)

**便捷功能**: 使用工厂函数快速创建注册中心

```python
# 支持的后端
create_registry("local", registry_path="./registry")  # 本地文件系统
create_registry("mlflow", tracking_uri="http://...")  # MLflow（需安装）
```

# ============================================================
# 使用工厂函数创建注册中心
# ============================================================
# create_registry() 是创建注册中心的便捷方法
# 支持不同的后端：local（本地文件系统）、mlflow（MLflow服务器）

# 创建临时目录
factory_dir = demo_dir + "_factory"

# 使用工厂函数创建本地注册中心
registry2 = create_registry("local", registry_path=factory_dir)

# 创建测试文件
test_model = Path(demo_dir) / "factory_model.pkl"
test_model.write_bytes(b"test model content")

# 注册模型
registry2.register_model(
    "test_model",
    str(test_model),
    metrics={"score": 0.88}
)

print("=" * 50)
print("工厂函数创建的注册中心")
print("=" * 50)
print("运行成功!")
print(f"存储路径: {factory_dir}")

# 清理
shutil.rmtree(factory_dir, ignore_errors=True)

In [ ]:
## 5. MLflow 集成 (可选)

**企业级方案**: MLflow 提供更强大的模型注册功能

```
MLflow 模型注册优势:
┌─────────────────────────────────────────────────────────────┐
│  ✓ Web UI 可视化管理                                        │
│  ✓ 团队协作支持                                             │
│  ✓ 模型服务部署                                             │
│  ✓ 与实验追踪集成                                           │
│  ✓ REST API 支持                                            │
└─────────────────────────────────────────────────────────────┘
```

# ============================================================
# MLflow 模型注册示例代码
# ============================================================
# 如果安装了 MLflow，可以使用更强大的模型注册功能

if MLFLOW_AVAILABLE:
    print("=" * 50)
    print("MLflow 模型注册使用示例")
    print("=" * 50)
    print("""
# 1. 创建 MLflow 注册中心
from model_registry import MLflowRegistry

registry = MLflowRegistry(
    tracking_uri="http://localhost:5000"  # MLflow 服务器地址
)

# 2. 从 MLflow run 注册模型
version = registry.register_model(
    name="my_model",
    model_path="runs:/run_id/model"  # MLflow run 中的模型路径
)

# 3. 转换阶段
registry.transition_stage("my_model", "1", ModelStage.PRODUCTION)

# 4. 加载生产模型进行推理
model = registry.load_model("my_model", stage=ModelStage.PRODUCTION)
predictions = model.predict(data)
""")
else:
    print("=" * 50)
    print("MLflow 未安装")
    print("=" * 50)
    print("安装命令: pip install mlflow")
    print("\n安装后可以使用:")
    print("  - Web UI 可视化管理模型")
    print("  - 团队协作和权限控制")
    print("  - 模型服务部署 (mlflow models serve)")
    print("  - REST API 访问")

In [ ]:
## 6. 清理演示目录

# ============================================================
# 清理演示目录
# ============================================================
# 删除演示过程中创建的临时文件和目录

shutil.rmtree(demo_dir, ignore_errors=True)
print("演示目录已清理")

In [ ]:
## 总结

本教程介绍了模型注册中心的核心功能：

```
核心概念回顾:
┌─────────────────────────────────────────────────────────────┐
│  ModelStage      → 模型生命周期阶段                         │
│  ModelVersion    → 版本数据结构                             │
│  ModelRegistry   → 注册中心核心类                           │
│                                                             │
│  核心操作:                                                   │
│  ├── register_model()     注册新版本                        │
│  ├── get_model()          获取模型                          │
│  ├── transition_stage()   阶段转换                          │
│  ├── list_models/versions 列出模型                          │
│  ├── search_models()      搜索模型                          │
│  ├── compare_versions()   比较版本                          │
│  └── delete_model()       删除模型                          │
└─────────────────────────────────────────────────────────────┘
```

### 最佳实践

| 实践 | 说明 |
|:-----|:-----|
| 语义化版本号 | 使用 major.minor.patch 格式 |
| 记录元数据 | 保存指标、参数、标签 |
| 阶段管理 | 先 Staging 测试，再 Production |
| 保留历史 | 归档旧版本用于回滚 |
| 按阶段获取 | 生产代码使用 stage 参数 |

### 下一步

- 学习 [03_Monitoring_tutorial](03_Monitoring_tutorial.ipynb) 了解生产监控
- 学习 [04_DriftDetection_tutorial](04_DriftDetection_tutorial.ipynb) 了解漂移检测

## 总结

本教程介绍了模型注册中心的核心功能：

1. **模型阶段**: Development → Staging → Production
2. **版本管理**: 注册、获取、列出版本
3. **阶段转换**: 自动归档旧版本
4. **搜索和比较**: 按名称、标签、阶段搜索
5. **MLflow 集成**: 企业级模型管理

### 最佳实践

- 使用语义化版本号 (major.minor.patch)
- 记录模型的指标和参数
- 使用标签组织模型
- 在推送到生产前进行充分测试
- 保留历史版本用于回滚